# UNIVERSIDAD EXTERNADO DE COLOMBIA · CIENCIA DE DATOS · 2026

## Modelos segmentados por grupo etario

**Predicción de la Estancia Hospitalaria con Aprendizaje Automático — SPARCS CY2021**

Autores: Daniel Vallejo Vecil · Juliana M.

---

Este cuaderno evalúa si ajustar modelos independientes por grupo etario mejora la predicción de la DEH
frente a un modelo único entrenado sobre la población completa. La comparación se realiza sobre las
**tres familias de algoritmos** empleadas en el resto del proyecto —regresión regularizada, Random Forest
y XGBoost con pérdida de Tweedie—, de modo que el efecto de la segmentación pueda evaluarse con
independencia del algoritmo elegido.

## 1. Fundamento

Un modelo único debe representar simultáneamente regímenes clínicos muy distintos: episodios neonatales,
hospitalizaciones pediátricas, atención obstétrica y patología crónica del adulto mayor responden a
determinantes diferentes y presentan distribuciones de estancia también distintas. La segmentación
permite que cada modelo especialice su estructura sobre una población homogénea, a costa de reducir el
número de observaciones disponibles para cada ajuste.

Jain et al. (2024) aportan evidencia directa sobre la misma fuente: obtienen un $R^2$ de 0,82 en la
población de recién nacidos frente a valores cercanos a 0,43 en el resto, y alcanzan 0,62 al restringir
el análisis a un único diagnóstico.

### 1.1 Diseño experimental

Se varía **una sola condición a la vez**. Cada familia se entrena en dos configuraciones —un modelo
global y un conjunto de modelos por segmento— manteniendo hiperparámetros idénticos entre ambas. Así,
cualquier diferencia observada dentro de una familia es atribuible a la segmentación, y cualquier
diferencia entre familias bajo la misma configuración es atribuible al algoritmo.

Todos los modelos son de **una sola etapa**. El esquema de doble valla evaluado en el cuaderno 08 se
mantiene fuera de este diseño de forma deliberada: combinarlo con la segmentación haría variar dos
factores simultáneamente e impediría atribuir el efecto a ninguno de los dos. Si la segmentación resulta
ventajosa, su combinación con el esquema en dos fases queda como extensión natural.

### 1.2 Advertencia metodológica sobre la comparación

Comparar directamente el $R^2$ de un segmento con el de otro, o con el del modelo global evaluado sobre
toda la población, es **incorrecto**. El coeficiente de determinación se define respecto de la varianza
del desenlace dentro del conjunto evaluado, de modo que un segmento con estancias homogéneas producirá
un $R^2$ bajo aunque su error absoluto sea pequeño.

La comparación válida es **pareada dentro de cada segmento**: el error del modelo global restringido a
ese segmento frente al del modelo específico entrenado sobre él. Para el juicio de conjunto se
recomponen las predicciones de todos los segmentos en un único vector, sobre el cual se calculan
métricas globales comparables con las de los cuadernos anteriores.

## 2. Configuración y carga

In [7]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import joblib, time, re

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.base import clone
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

PROJECT_ROOT = Path.cwd().parent
GOLD_PATH = PROJECT_ROOT / "data" / "3_gold"
MODELOS_PATH = PROJECT_ROOT / "modelos"
TABLAS_PATH = PROJECT_ROOT / "tablas"
MODELOS_PATH.mkdir(parents=True, exist_ok=True)
TABLAS_PATH.mkdir(parents=True, exist_ok=True)

NOTEBOOK_NUM = "09"


def guardar_tabla(tabla, nombre):
    if isinstance(tabla, pd.Series):
        tabla = tabla.to_frame()
    archivo = TABLAS_PATH / f"{NOTEBOOK_NUM}_{nombre}.csv"
    tabla.to_csv(archivo, encoding="utf-8-sig")
    print(f"Tabla guardada: {archivo.resolve()}")


# El split viene fijado desde 03_FeatureEngineering. NO se vuelve a particionar.
X_train = pd.read_parquet(GOLD_PATH / "X_train_sparcs.parquet")
X_test = pd.read_parquet(GOLD_PATH / "X_test_sparcs.parquet")
y_train = pd.read_parquet(GOLD_PATH / "y_train_sparcs.parquet")["Length of Stay"]
y_test = pd.read_parquet(GOLD_PATH / "y_test_sparcs.parquet")["Length of Stay"]

print(f"X_train: {X_train.shape}   X_test: {X_test.shape}")

X_train: (1672271, 189)   X_test: (418068, 189)


In [8]:
def sanear(nombres):
    """Convierte nombres de columna a caracteres seguros, evitando colisiones."""
    limpios, vistos = [], {}
    for n in nombres:
        s = re.sub(r"[^0-9a-zA-Z_]+", "_", str(n)).strip("_")
        if not s:
            s = "col"
        if s[0].isdigit():
            s = "v_" + s
        if s in vistos:
            vistos[s] += 1
            s = f"{s}__{vistos[s]}"
        else:
            vistos[s] = 0
        limpios.append(s)
    return limpios


NOMBRES_ORIGINALES = list(X_train.columns)
nombres_saneados = sanear(NOMBRES_ORIGINALES)
MAPA_NOMBRES = dict(zip(nombres_saneados, NOMBRES_ORIGINALES))
X_train.columns = nombres_saneados
X_test.columns = nombres_saneados

print(f"Columnas: {len(nombres_saneados)}  |  duplicadas: "
      f"{len(nombres_saneados) - len(set(nombres_saneados))}")
joblib.dump(MAPA_NOMBRES, MODELOS_PATH / "09_mapa_nombres_columnas.joblib")

Columnas: 189  |  duplicadas: 0


["c:\\Users\\julia\\OneDrive - Universidad Externado de Colombia\\Shortcuts\\Daniel Humberto Vallejo Vecil's files - Tesis\\SPARCS\\modelos\\09_mapa_nombres_columnas.joblib"]

## 3. Reconstrucción del grupo etario

La variable `Age Group` fue codificada mediante indicadores binarios con supresión de la primera
categoría, de modo que la categoría de referencia —el grupo de menor edad— no tiene columna propia y se
identifica por la ausencia de todas las demás.

In [9]:
cols_edad = [col for col in X_train.columns if col.lower().startswith("age_group")]
print("Columnas indicadoras de grupo etario detectadas:")
for col in cols_edad:
    print(f"  {col}   (original: {MAPA_NOMBRES[col]})")

if not cols_edad:
    raise ValueError("No se detectaron columnas de grupo etario. Revisar el prefijo.")

ETIQUETA_REFERENCIA = "0 to 17"   # categoría suprimida; ajustar si difiere


def reconstruir_grupo(X):
    """Recupera la categoría de grupo etario a partir de los indicadores binarios."""
    etiquetas = pd.Series(ETIQUETA_REFERENCIA, index=X.index, dtype=object)
    for col in cols_edad:
        etiquetas[X[col] == 1] = MAPA_NOMBRES[col].split("_", 1)[-1]
    return etiquetas


grupo_train = reconstruir_grupo(X_train)
grupo_test = reconstruir_grupo(X_test)

conteo = pd.DataFrame({
    "n_train": grupo_train.value_counts(),
    "pct_train": grupo_train.value_counts(normalize=True) * 100,
    "n_test": grupo_test.value_counts(),
}).sort_index()
display(conteo.round(2))

Columnas indicadoras de grupo etario detectadas:
  Age_Group_18_to_29   (original: Age Group_18 to 29)
  Age_Group_30_to_49   (original: Age Group_30 to 49)
  Age_Group_50_to_69   (original: Age Group_50 to 69)
  Age_Group_70_or_Older   (original: Age Group_70 or Older)


,n_train,pct_train,n_test
0 to 17,226075,13.52,56873
18 to 29,150130,8.98,37511
30 to 49,329091,19.68,82222
50 to 69,467520,27.96,116758
70 or Older,499455,29.87,124704


## 4. La estancia difiere entre grupos etarios

Este diagnóstico determina si la segmentación tiene sentido antes de entrenar ningún modelo. Si la
distribución del desenlace fuera equivalente entre grupos, no habría razón para esperar que modelos
separados capturen estructuras distintas.

La columna `pct_larga` es además relevante para una decisión futura: indica si la valla de dos días del
cuaderno 08 tendría el mismo significado en todos los grupos, cuestión a resolver antes de combinar
segmentación y esquema en dos fases.

In [10]:
perfil = pd.DataFrame({
    "n": grupo_train.value_counts(),
    "media": y_train.groupby(grupo_train).mean(),
    "mediana": y_train.groupby(grupo_train).median(),
    "desv_std": y_train.groupby(grupo_train).std(),
    "p75": y_train.groupby(grupo_train).quantile(0.75),
    "p95": y_train.groupby(grupo_train).quantile(0.95),
    "asimetria": y_train.groupby(grupo_train).skew(),
    "pct_larga": (y_train > 2).groupby(grupo_train).mean() * 100,
}).sort_index()

display(perfil.round(2))
guardar_tabla(perfil, "PerfilDEHPorGrupoEtario")

print("\nLa varianza del desenlace difiere entre grupos: por eso el R2 de un segmento")
print("NO es comparable con el de otro. Ver advertencia de la Sección 1.2.")

,n,media,mediana,desv_std,p75,p95,asimetria,pct_larga
0 to 17,226075,4.05,2.0,9.10,3.0,13.0,7.97,33.30
18 to 29,150130,4.54,3.0,7.18,4.0,14.0,6.95,51.58
30 to 49,329091,4.87,3.0,7.60,5.0,16.0,6.63,54.55
50 to 69,467520,6.54,4.0,9.13,7.0,21.0,5.11,66.62
70 or Older,499455,6.70,4.0,7.87,8.0,20.0,4.90,73.75


Tabla guardada: C:\Users\julia\OneDrive - Universidad Externado de Colombia\Shortcuts\Daniel Humberto Vallejo Vecil's files - Tesis\SPARCS\tablas\09_PerfilDEHPorGrupoEtario.csv

La varianza del desenlace difiere entre grupos: por eso el R2 de un segmento
NO es comparable con el de otro. Ver advertencia de la Sección 1.2.


## 5. Familias de algoritmos y modelos globales de referencia

Las tres familias se definen con los mismos hiperparámetros empleados en los cuadernos anteriores, y
cada una se entrena primero sobre la población completa. Estas predicciones constituyen la referencia
contra la cual se evalúa la segmentación, familia por familia.

In [11]:
FRACCION = 0.30   # poner 1.0 para la ejecución definitiva
SEMILLA = 42

if FRACCION < 1.0:
    idx = X_train.sample(frac=FRACCION, random_state=SEMILLA).index
    X_tr, y_tr, grupo_tr = X_train.loc[idx], y_train.loc[idx], grupo_train.loc[idx]
else:
    X_tr, y_tr, grupo_tr = X_train, y_train, grupo_train

print(f"Entrenamiento: {X_tr.shape[0]:,} filas   |   Prueba: {X_test.shape[0]:,} filas")

FAMILIAS = {
    "Ridge": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(
        n_estimators=200, max_depth=20, random_state=SEMILLA, n_jobs=-1),
    "XGBoost Tweedie": XGBRegressor(
        objective="reg:tweedie", tweedie_variance_power=1.5,
        n_estimators=400, max_depth=8, learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8, tree_method="hist", random_state=SEMILLA, n_jobs=-1),
}


def metricas(y_real, y_pred):
    y_pred = np.clip(y_pred, 1, None)
    return {"MAE": mean_absolute_error(y_real, y_pred),
            "RMSE": float(np.sqrt(mean_squared_error(y_real, y_pred))),
            "R2": r2_score(y_real, y_pred)}


pred_global, modelos_globales, filas_glob = {}, {}, []

for fam, plantilla in FAMILIAS.items():
    t0 = time.time()
    modelo = clone(plantilla)
    modelo.fit(X_tr, y_tr)
    p = np.clip(modelo.predict(X_test), 1, None)

    pred_global[fam] = p
    modelos_globales[fam] = modelo
    filas_glob.append({"familia": fam, "configuracion": "Global",
                       **metricas(y_test, p), "segundos": round(time.time() - t0, 1)})
    print(f"{fam:18s}  MAE={filas_glob[-1]['MAE']:.3f}  R2={filas_glob[-1]['R2']:.4f}  "
          f"({filas_glob[-1]['segundos']}s)")
    joblib.dump(modelo, MODELOS_PATH / f"09_global_{re.sub(r'[^0-9a-zA-Z]+', '_', fam)}.joblib")

glob_df = pd.DataFrame(filas_glob)
display(glob_df.round(4))
guardar_tabla(glob_df, "ModelosGlobalesPorFamilia")

Entrenamiento: 501,681 filas   |   Prueba: 418,068 filas
Ridge               MAE=3.267  R2=0.3837  (1.9s)
Random Forest       MAE=2.994  R2=0.4263  (1717.3s)
XGBoost Tweedie     MAE=2.910  R2=0.4535  (55.0s)


,familia,configuracion,MAE,RMSE,R2,segundos
0,Ridge,Global,3.2673,6.5174,0.3837,1.9
1,Random Forest,Global,2.9942,6.2883,0.4263,1717.3
2,XGBoost Tweedie,Global,2.9103,6.1371,0.4535,55.0


Tabla guardada: C:\Users\julia\OneDrive - Universidad Externado de Colombia\Shortcuts\Daniel Humberto Vallejo Vecil's files - Tesis\SPARCS\tablas\09_ModelosGlobalesPorFamilia.csv


## 6. Modelos por segmento y comparación pareada

Para cada familia y cada grupo etario se entrena un modelo independiente con los mismos hiperparámetros
del modelo global correspondiente. Se contrasta después, dentro de cada segmento, el desempeño del
modelo específico frente al del modelo global restringido a ese mismo subconjunto.

Los segmentos con menos observaciones que `MIN_OBS` se respaldan con la predicción del modelo global, de
modo que el vector de predicciones segmentadas quede completo sobre el conjunto de prueba.

In [12]:
MIN_OBS = 1000

pred_segmentada = {fam: pd.Series(np.nan, index=X_test.index, dtype=float) for fam in FAMILIAS}
filas_seg = []

for fam, plantilla in FAMILIAS.items():
    print(f"\n=== Familia: {fam} ===")
    for g in sorted(grupo_tr.unique()):
        m_tr = (grupo_tr == g).to_numpy()
        m_te = (grupo_test == g).to_numpy()

        if m_tr.sum() < MIN_OBS or m_te.sum() == 0:
            print(f"  {g:14s} omitido (n_train={m_tr.sum()}, n_test={m_te.sum()})")
            pred_segmentada[fam][m_te] = pred_global[fam][m_te]
            continue

        t0 = time.time()
        modelo = clone(plantilla)
        modelo.fit(X_tr[m_tr], y_tr[m_tr])
        p_seg = np.clip(modelo.predict(X_test[m_te]), 1, None)
        pred_segmentada[fam][m_te] = p_seg

        y_seg = y_test[m_te]
        m_glob = metricas(y_seg, pred_global[fam][m_te])
        m_seg = metricas(y_seg, p_seg)

        filas_seg.append({
            "familia": fam, "segmento": g,
            "n_train": int(m_tr.sum()), "n_test": int(m_te.sum()),
            "MAE_global": m_glob["MAE"], "MAE_segmentado": m_seg["MAE"],
            "mejora_MAE_pct": (m_glob["MAE"] - m_seg["MAE"]) / m_glob["MAE"] * 100,
            "R2_global": m_glob["R2"], "R2_segmentado": m_seg["R2"],
            "delta_R2": m_seg["R2"] - m_glob["R2"],
        })
        print(f"  {g:14s} MAE {m_glob['MAE']:.3f} -> {m_seg['MAE']:.3f}  "
              f"({filas_seg[-1]['mejora_MAE_pct']:+.1f}%)   R2 {m_glob['R2']:.4f} -> "
              f"{m_seg['R2']:.4f}   ({time.time()-t0:.0f}s)")

        nombre_archivo = re.sub(r"[^0-9a-zA-Z]+", "_", f"{fam}_{g}")
        joblib.dump(modelo, MODELOS_PATH / f"09_seg_{nombre_archivo}.joblib")

comp_seg = pd.DataFrame(filas_seg)
display(comp_seg.round(4))
guardar_tabla(comp_seg, "ComparacionPorSegmentoYFamilia")


=== Familia: Ridge ===
  0 to 17        MAE 2.128 -> 2.090  (+1.8%)   R2 0.5789 -> 0.6033   (1s)
  18 to 29       MAE 2.590 -> 2.552  (+1.5%)   R2 0.3411 -> 0.3424   (1s)
  30 to 49       MAE 2.830 -> 2.793  (+1.3%)   R2 0.3575 -> 0.3604   (1s)
  50 to 69       MAE 3.836 -> 3.830  (+0.1%)   R2 0.3551 -> 0.3581   (1s)
  70 or Older    MAE 3.747 -> 3.725  (+0.6%)   R2 0.2973 -> 0.3050   (1s)

=== Familia: Random Forest ===
  0 to 17        MAE 1.781 -> 1.776  (+0.3%)   R2 0.6242 -> 0.6246   (82s)
  18 to 29       MAE 2.380 -> 2.401  (-0.9%)   R2 0.3657 -> 0.3297   (57s)
  30 to 49       MAE 2.548 -> 2.590  (-1.7%)   R2 0.4031 -> 0.3698   (175s)
  50 to 69       MAE 3.548 -> 3.606  (-1.7%)   R2 0.3929 -> 0.3752   (327s)
  70 or Older    MAE 3.508 -> 3.541  (-0.9%)   R2 0.3502 -> 0.3409   (342s)

=== Familia: XGBoost Tweedie ===
  0 to 17        MAE 1.750 -> 1.698  (+2.9%)   R2 0.6416 -> 0.6284   (7s)
  18 to 29       MAE 2.242 -> 2.241  (+0.1%)   R2 0.4002 -> 0.3647   (6s)
  30 to 49    

,familia,segmento,n_train,n_test,MAE_global,MAE_segmentado,mejora_MAE_pct,R2_global,R2_segmentado,delta_R2
0,Ridge,0 to 17,67932,56873,2.1282,2.0897,1.8114,0.5789,0.6033,0.0244
1,Ridge,18 to 29,44915,37511,2.5903,2.5517,1.4879,0.3411,0.3424,0.0013
2,Ridge,30 to 49,98699,82222,2.8299,2.7934,1.2889,0.3575,0.3604,0.0028
3,Ridge,50 to 69,140201,116758,3.8356,3.8300,0.1471,0.3551,0.3581,0.0031
4,Ridge,70 or Older,149934,124704,3.7467,3.7247,0.5892,0.2973,0.3050,0.0077
5,Random Forest,0 to 17,67932,56873,1.7809,1.7760,0.2762,0.6242,0.6246,0.0005
6,Random Forest,18 to 29,44915,37511,2.3798,2.4012,-0.9026,0.3657,0.3297,-0.0359
7,Random Forest,30 to 49,98699,82222,2.5479,2.5902,-1.6595,0.4031,0.3698,-0.0333
8,Random Forest,50 to 69,140201,116758,3.5476,3.6063,-1.6549,0.3929,0.3752,-0.0178
9,Random Forest,70 or Older,149934,124704,3.5085,3.5409,-0.9237,0.3502,0.3409,-0.0093


Tabla guardada: C:\Users\julia\OneDrive - Universidad Externado de Colombia\Shortcuts\Daniel Humberto Vallejo Vecil's files - Tesis\SPARCS\tablas\09_ComparacionPorSegmentoYFamilia.csv


In [13]:
# Vista cruzada: mejora porcentual del MAE, por segmento y familia
pivote = comp_seg.pivot(index="segmento", columns="familia", values="mejora_MAE_pct")
print("Mejora porcentual del MAE por segmentación (positivo = la segmentación aporta):")
display(pivote.round(2))
guardar_tabla(pivote, "PivoteMejoraMAE")

Mejora porcentual del MAE por segmentación (positivo = la segmentación aporta):


familia,Random Forest,Ridge,XGBoost Tweedie
segmento,,,
0 to 17,0.28,1.81,2.93
18 to 29,-0.90,1.49,0.07
30 to 49,-1.66,1.29,0.21
50 to 69,-1.65,0.15,0.09
70 or Older,-0.92,0.59,0.25


Tabla guardada: C:\Users\julia\OneDrive - Universidad Externado de Colombia\Shortcuts\Daniel Humberto Vallejo Vecil's files - Tesis\SPARCS\tablas\09_PivoteMejoraMAE.csv


Un patrón consistente entre familias —todas mejoran en los mismos segmentos y empeoran en los mismos—
indica que el efecto proviene de la estructura de la población y no de una interacción particular con
algún algoritmo. Si por el contrario solo una familia se beneficia, lo que la segmentación está
corrigiendo es una limitación de ese modelo concreto.

## 7. Juicio de conjunto por familia

Las predicciones de todos los segmentos se recomponen en un vector único sobre el conjunto de prueba
completo, lo que permite comparar cada configuración segmentada con su propio modelo global y con los
resultados de los cuadernos anteriores.

In [14]:
filas_final = []
for fam in FAMILIAS:
    assert pred_segmentada[fam].notna().all(), f"Predicciones incompletas en {fam}"
    m_g = metricas(y_test, pred_global[fam])
    m_s = metricas(y_test, pred_segmentada[fam].to_numpy())
    filas_final.append({
        "familia": fam,
        "MAE_global": m_g["MAE"], "MAE_segmentado": m_s["MAE"],
        "delta_MAE": m_g["MAE"] - m_s["MAE"],
        "R2_global": m_g["R2"], "R2_segmentado": m_s["R2"],
        "delta_R2": m_s["R2"] - m_g["R2"],
    })

final_df = pd.DataFrame(filas_final)
display(final_df.round(4))
guardar_tabla(final_df, "ResumenGlobalVsSegmentado")

mejor = final_df.sort_values("MAE_segmentado").iloc[0]
print(f"\nMejor configuración segmentada: {mejor['familia']}  "
      f"(MAE={mejor['MAE_segmentado']:.3f}, R2={mejor['R2_segmentado']:.4f})")

if (final_df["delta_R2"] < 0.005).all() and (final_df["delta_MAE"] < 0.05).all():
    print("\n>>> La ganancia es marginal en las tres familias. La segmentación no justifica")
    print(">>> mantener múltiples modelos ni renunciar a una interpretación SHAP unificada.")

,familia,MAE_global,MAE_segmentado,delta_MAE,R2_global,R2_segmentado,delta_R2
0,Ridge,3.2673,3.2433,0.0240,0.3837,0.3910,0.0073
1,Random Forest,2.9942,3.0298,-0.0356,0.4263,0.4100,-0.0163
2,XGBoost Tweedie,2.9103,2.8987,0.0116,0.4535,0.4428,-0.0108


Tabla guardada: C:\Users\julia\OneDrive - Universidad Externado de Colombia\Shortcuts\Daniel Humberto Vallejo Vecil's files - Tesis\SPARCS\tablas\09_ResumenGlobalVsSegmentado.csv

Mejor configuración segmentada: XGBoost Tweedie  (MAE=2.899, R2=0.4428)


## 8. Sub-segmentación: episodios neonatales

El grupo de menor edad concentra poblaciones heterogéneas: recién nacidos sanos, prematuros y patología
pediátrica general. Jain et al. (2024) reportan que la mejora sustancial se concentra específicamente en
los recién nacidos y no en el grupo etario completo. Esta sección verifica si la ganancia observada —de
haberla— proviene de ese subgrupo, empleando las tres familias.

In [15]:
cols_newborn = [col for col in X_train.columns
                if "newborn" in col.lower() and col.lower().startswith("type_of_admission")]

if not cols_newborn:
    print("No se detectó el indicador de admisión neonatal. Columnas candidatas:")
    print([c for c in X_train.columns if "type_of_admission" in c.lower()])
else:
    COL_NB = cols_newborn[0]
    print(f"Indicador neonatal: {COL_NB}  (original: {MAPA_NOMBRES[COL_NB]})\n")

    nb_tr = (X_tr[COL_NB] == 1).to_numpy()
    nb_te = (X_test[COL_NB] == 1).to_numpy()
    print(f"Episodios neonatales — entrenamiento: {nb_tr.sum():,}  |  prueba: {nb_te.sum():,}\n")

    filas_nb = []
    for fam, plantilla in FAMILIAS.items():
        for etiqueta, m_tr_sub, m_te_sub in [("Neonatales", nb_tr, nb_te),
                                             ("No neonatales", ~nb_tr, ~nb_te)]:
            modelo = clone(plantilla)
            modelo.fit(X_tr[m_tr_sub], y_tr[m_tr_sub])
            p = np.clip(modelo.predict(X_test[m_te_sub]), 1, None)
            m_glob = metricas(y_test[m_te_sub], pred_global[fam][m_te_sub])
            m_sub = metricas(y_test[m_te_sub], p)
            filas_nb.append({
                "familia": fam, "subgrupo": etiqueta, "n_test": int(m_te_sub.sum()),
                "dias_medios": float(y_test[m_te_sub].mean()),
                "MAE_global": m_glob["MAE"], "MAE_especifico": m_sub["MAE"],
                "R2_global": m_glob["R2"], "R2_especifico": m_sub["R2"],
            })
            print(f"{fam:18s} {etiqueta:16s} MAE {m_glob['MAE']:.3f} -> {m_sub['MAE']:.3f}   "
                  f"R2 {m_glob['R2']:.4f} -> {m_sub['R2']:.4f}")

    nb_df = pd.DataFrame(filas_nb)
    display(nb_df.round(4))
    guardar_tabla(nb_df, "SubsegmentacionNeonatal")

Indicador neonatal: Type_of_Admission_Newborn  (original: Type of Admission_Newborn)

Episodios neonatales — entrenamiento: 46,552  |  prueba: 38,825

Ridge              Neonatales       MAE 1.568 -> 1.455   R2 0.6881 -> 0.7166
Ridge              No neonatales    MAE 3.441 -> 3.436   R2 0.3479 -> 0.3480
Random Forest      Neonatales       MAE 1.247 -> 1.261   R2 0.7296 -> 0.7236
Random Forest      No neonatales    MAE 3.173 -> 3.182   R2 0.3909 -> 0.3854
XGBoost Tweedie    Neonatales       MAE 1.243 -> 1.235   R2 0.7315 -> 0.7066
XGBoost Tweedie    No neonatales    MAE 3.081 -> 3.078   R2 0.4210 -> 0.4218


,familia,subgrupo,n_test,dias_medios,MAE_global,MAE_especifico,R2_global,R2_especifico
0,Ridge,Neonatales,38825,3.4393,1.5679,1.4551,0.6881,0.7166
1,Ridge,No neonatales,379243,5.9554,3.4413,3.4364,0.3479,0.3480
2,Random Forest,Neonatales,38825,3.4393,1.2473,1.2614,0.7296,0.7236
3,Random Forest,No neonatales,379243,5.9554,3.1730,3.1819,0.3909,0.3854
4,XGBoost Tweedie,Neonatales,38825,3.4393,1.2426,1.2346,0.7315,0.7066
5,XGBoost Tweedie,No neonatales,379243,5.9554,3.0810,3.0783,0.4210,0.4218


Tabla guardada: C:\Users\julia\OneDrive - Universidad Externado de Colombia\Shortcuts\Daniel Humberto Vallejo Vecil's files - Tesis\SPARCS\tablas\09_SubsegmentacionNeonatal.csv


Si el $R^2$ del modelo neonatal específico resulta sustancialmente superior al del resto, se replica el
hallazgo de Jain et al. (2024) sobre esta misma fuente, lo que constituye validación externa del trabajo
y sugiere que la heterogeneidad relevante no es la edad sino el tipo de episodio.

Conviene recordar que `Birth Weight` fue excluida en la capa Silver por su proporción de nulos, pese a
ser previsiblemente el predictor más informativo de ese subgrupo. Es esperable que el desempeño del
segmento neonatal mejore al recuperarla con una categoría explícita para los episodios no neonatales.

## 9. ¿La pérdida en segmentos pequeños se debe a la falta de datos?

Cuando un segmento rinde peor que el modelo global, caben dos explicaciones: que su estructura no difiera
lo suficiente como para justificar un modelo propio, o que el número de observaciones disponibles sea
insuficiente. Esta sección las distingue entrenando con fracciones crecientes del propio segmento: si la
curva aún no se ha estabilizado, el problema es de tamaño muestral.

In [16]:
EJECUTAR_CURVAS = False   # activar solo para los segmentos de interés
FAMILIA_CURVA = "XGBoost Tweedie"

if EJECUTAR_CURVAS:
    SEGMENTOS_CURVA = sorted(grupo_tr.unique())[:3]
    FRACCIONES = [0.1, 0.25, 0.5, 0.75, 1.0]

    filas_curva = []
    for g in SEGMENTOS_CURVA:
        m_tr = (grupo_tr == g).to_numpy()
        m_te = (grupo_test == g).to_numpy()
        if m_tr.sum() < MIN_OBS:
            continue
        Xg, yg = X_tr[m_tr], y_tr[m_tr]

        for f in FRACCIONES:
            sub = Xg.sample(frac=f, random_state=SEMILLA).index
            modelo = clone(FAMILIAS[FAMILIA_CURVA])
            modelo.fit(Xg.loc[sub], yg.loc[sub])
            p = np.clip(modelo.predict(X_test[m_te]), 1, None)
            filas_curva.append({"segmento": g, "fraccion": f, "n": len(sub),
                                **metricas(y_test[m_te], p)})
            print(f"{g:14s} frac={f:.2f}  n={len(sub):>8,}  MAE={filas_curva[-1]['MAE']:.3f}")

    curva_df = pd.DataFrame(filas_curva)
    display(curva_df.round(4))
    guardar_tabla(curva_df, "CurvasAprendizajePorSegmento")
else:
    print("Curvas desactivadas. Cambiar EJECUTAR_CURVAS = True para ejecutarlas.")

Curvas desactivadas. Cambiar EJECUTAR_CURVAS = True para ejecutarlas.


## 10. Conclusiones del cuaderno

Elementos a documentar en el Capítulo 4:

1. **El perfil de la DEH por grupo etario** (Sección 4). Justifica o descarta la segmentación antes de
   entrenar, y explica por qué el $R^2$ no es comparable entre segmentos.
2. **La consistencia del efecto entre familias** (Sección 6). Si las tres familias mejoran en los mismos
   segmentos, el efecto proviene de la estructura de la población; si solo una lo hace, la segmentación
   está corrigiendo una limitación de ese algoritmo concreto.
3. **El juicio de conjunto por familia** (Sección 7). Determina si la segmentación mejora la predicción
   sobre la población completa. Una ganancia inferior a 0,005 de $R^2$ no justifica mantener varios
   modelos ni renunciar a una interpretación SHAP unificada.
4. **La sub-segmentación neonatal** (Sección 8). Si reproduce el contraste reportado por Jain et al.
   (2024), constituye validación externa y sugiere que la heterogeneidad relevante es el tipo de episodio
   y no la edad.
5. **La implicación para el análisis de interpretabilidad.** Si la segmentación resulta ganadora, el
   análisis SHAP debe realizarse por segmento, y la comparación de los ordenamientos entre grupos pasa a
   ser un resultado por derecho propio: permite establecer si los determinantes de la estancia son los
   mismos en toda la población o cambian con la edad.
6. **La columna `pct_larga` de la Sección 4** como insumo para una eventual combinación con el esquema de
   doble valla: si la proporción de estancias prolongadas difiere marcadamente entre grupos, el umbral de
   dos días no tendría el mismo significado en todos y requeriría justificación adicional.

**Tablas exportadas:** `09_PerfilDEHPorGrupoEtario`, `09_ModelosGlobalesPorFamilia`,
`09_ComparacionPorSegmentoYFamilia`, `09_PivoteMejoraMAE`, `09_ResumenGlobalVsSegmentado`,
`09_SubsegmentacionNeonatal`, `09_CurvasAprendizajePorSegmento`.